In [6]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
BLUE, ORANGE, INK, MUTED, SURFACE = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#fcfcfb"

In [8]:
#1) Load & clean (Pandas)
df = pd.read_csv("yerevan_coffee_weather.csv", parse_dates=["date"])
print("Raw shape:", df.shape)
print("Missing values:\n", df.isna().sum()[lambda s: s > 0], "\n")

bad_temp = (df["temp_c"] > 50) | (df["temp_c"] < -40)
print("Sensor errors (temp out of range):", int(bad_temp.sum()))
df.loc[bad_temp, "temp_c"] = np.nan


Raw shape: (500, 7)
Missing values:
 temp_c         6
coffee_cups    3
dtype: int64 

Sensor errors (temp out of range): 3


In [9]:
# Fill gaps: temperature by interpolation in time, coffee by weekday median
df = df.sort_values("date").reset_index(drop=True)
df["temp_c"] = df["temp_c"].interpolate(limit_direction="both")
df["coffee_cups"] = df["coffee_cups"].fillna(
    df.groupby("weekday")["coffee_cups"].transform("median")).round().astype(int)

df["is_weekend"] = df["weekday"].isin(["Saturday", "Sunday"]).astype(int)
df["temp_band"] = pd.cut(df["temp_c"], [-100, 5, 15, 25, 100],
                         labels=["<5°C", "5–15°C", "15–25°C", ">25°C"])
df.to_csv("yerevan_coffee_clean.csv", index=False)
print("Clean shape:", df.shape, "| remaining NaN:", int(df.isna().sum().sum()), "\n")

Clean shape: (500, 9) | remaining NaN: 0 



In [10]:
# 2) Correlations 
corr = df[["temp_c", "is_rainy", "precip_mm", "is_weekend",
           "coffee_cups", "cafe_checkins"]].corr().round(2)
print("Correlation matrix:\n", corr, "\n")

Correlation matrix:
                temp_c  is_rainy  precip_mm  is_weekend  coffee_cups  \
temp_c           1.00      0.08       0.09       -0.02        -0.77   
is_rainy         0.08      1.00       0.81        0.02         0.33   
precip_mm        0.09      0.81       1.00        0.02         0.26   
is_weekend      -0.02      0.02       0.02        1.00        -0.23   
coffee_cups     -0.77      0.33       0.26       -0.23         1.00   
cafe_checkins    0.29      0.44       0.34        0.64        -0.23   

               cafe_checkins  
temp_c                  0.29  
is_rainy                0.44  
precip_mm               0.34  
is_weekend              0.64  
coffee_cups            -0.23  
cafe_checkins           1.00   



In [11]:
# 3) SQL (SQLite) 
con = sqlite3.connect("coffee.db")
df.assign(date=df["date"].dt.strftime("%Y-%m-%d"),
          temp_band=df["temp_band"].astype(str)).to_sql("daily", con, if_exists="replace", index=False)


500

In [12]:
queries = {
    "Q1. Avg coffee on days colder than 15°C":
        "SELECT ROUND(AVG(coffee_cups),1) AS avg_coffee, COUNT(*) AS days FROM daily WHERE temp_c < 15;",
    "Q2. Rainy vs dry days":
        """SELECT CASE is_rainy WHEN 1 THEN 'Rainy' ELSE 'Dry' END AS day_type,
                  ROUND(AVG(coffee_cups),1) AS avg_coffee,
                  ROUND(AVG(cafe_checkins),1) AS avg_checkins, COUNT(*) AS days
           FROM daily GROUP BY is_rainy;""",
    "Q3. Weekday ranking (avg check-ins)":
        """SELECT weekday, ROUND(AVG(cafe_checkins),1) AS avg_checkins, ROUND(AVG(coffee_cups),1) AS avg_coffee
           FROM daily GROUP BY weekday ORDER BY avg_checkins DESC;""",
    "Q4. Temperature bands with >=50 days (HAVING)":
        """SELECT temp_band, COUNT(*) AS days, ROUND(AVG(coffee_cups),1) AS avg_coffee
           FROM daily GROUP BY temp_band HAVING COUNT(*) >= 50 ORDER BY avg_coffee DESC;""",
    "Q5. Month with highest avg coffee":
        """SELECT SUBSTR(date,1,7) AS month, ROUND(AVG(coffee_cups),1) AS avg_coffee
           FROM daily GROUP BY month ORDER BY avg_coffee DESC LIMIT 3;""",
    "Q6. Cold AND rainy days vs everything else":
        """SELECT CASE WHEN temp_c < 10 AND is_rainy = 1 THEN 'Cold + rain' ELSE 'Other' END AS grp,
                  ROUND(AVG(coffee_cups),1) AS avg_coffee, COUNT(*) AS days
           FROM daily GROUP BY grp;""",
}

In [13]:
sql_out = []
for title, q in queries.items():
    res = pd.read_sql(q, con)
    print(title); print(res.to_string(index=False)); print()
    sql_out.append((title, q, res))
con.close()

with open("queries.sql", "w") as f:
    for title, q, _ in sql_out:
        f.write(f"-- {title}\n{q}\n\n")


Q1. Avg coffee on days colder than 15°C
 avg_coffee  days
      281.7   314

Q2. Rainy vs dry days
day_type  avg_coffee  avg_checkins  days
     Dry       253.9         147.3   432
   Rainy       294.6         183.2    68

Q3. Weekday ranking (avg check-ins)
  weekday  avg_checkins  avg_coffee
 Saturday         182.8       247.2
   Sunday         178.6       240.2
   Monday         143.7       269.2
 Thursday         142.1       262.9
   Friday         140.9       267.4
  Tuesday         140.7       264.1
Wednesday         137.1       264.6

Q4. Temperature bands with >=50 days (HAVING)
temp_band  days  avg_coffee
     <5°C   199       289.9
   5–15°C   118       266.7
  15–25°C   119       234.2
    >25°C    64       198.0

Q5. Month with highest avg coffee
  month  avg_coffee
2025-01       297.9
2025-12       296.8
2025-02       294.6

Q6. Cold AND rainy days vs everything else
        grp  avg_coffee  days
Cold + rain       323.0    25
      Other       256.1   475



In [15]:
#4) Charts (Seaborn) 
sns.set_theme(style="whitegrid", rc={
    "axes.facecolor": SURFACE, "figure.facecolor": SURFACE, "grid.color": "#e6e5e1",
    "axes.edgecolor": "#c9c8c2", "text.color": INK, "axes.labelcolor": MUTED,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 11})

In [16]:
# Chart 1: temperature vs coffee, rainy vs dry
fig, ax = plt.subplots(figsize=(9, 5.5))
for rainy, color, label in [(0, BLUE, "Dry day"), (1, ORANGE, "Rainy day")]:
    sub = df[df.is_rainy == rainy]
    ax.scatter(sub.temp_c, sub.coffee_cups, s=22, color=color, alpha=0.75,
               edgecolor=SURFACE, linewidth=0.8, label=label)
    m, b = np.polyfit(sub.temp_c, sub.coffee_cups, 1)
    xs = np.linspace(sub.temp_c.min(), sub.temp_c.max(), 50)
    ax.plot(xs, m * xs + b, color=color, lw=2)
r = corr.loc["temp_c", "coffee_cups"]
ax.set_title(f"Colder days sell more coffee, and rain adds more (r = {r})",
             loc="left", fontsize=14, fontweight="bold", color=INK)
ax.set_xlabel("Temperature, °C"); ax.set_ylabel("Coffee cups per day")
ax.legend(frameon=False, loc="upper right")
sns.despine()
fig.tight_layout(); fig.savefig("chart1_temp_vs_coffee.png", dpi=160); plt.close(fig)

# Chart 2: avg check-ins by weekday
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
wk = df.groupby("weekday")["cafe_checkins"].mean().reindex(order)
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(wk.index.str[:3], wk.values, color=[BLUE] * 5 + [ORANGE] * 2, width=0.6)
for bar, v in zip(bars, wk.values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 2, f"{v:.0f}", ha="center", color=INK, fontsize=10)
ax.set_title("Weekends fill the cafés", loc="left", fontsize=14, fontweight="bold", color=INK)
ax.set_ylabel("Average café check-ins"); ax.set_ylim(0, wk.max() * 1.15)
sns.despine()
fig.tight_layout(); fig.savefig("chart2_weekday_checkins.png", dpi=160); plt.close(fig)

# Chart 3: correlation heatmap (single hue sequential + diverging-ish via blue/orange)
fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, cmap=sns.diverging_palette(40, 255, s=85, l=55, center="light", as_cmap=True),
            vmin=-1, vmax=1, center=0, linewidths=2, linecolor=SURFACE, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation matrix", loc="left", fontsize=14, fontweight="bold", color=INK)
plt.xticks(rotation=30, ha="right")
fig.tight_layout(); fig.savefig("chart3_correlation_heatmap.png", dpi=160); plt.close(fig)

print("Charts saved.")


Charts saved.
